# Notebook note

The paper plots are rebuilt from saved JSON results. I use this notebook when rerunning the CIFAR-100 ViT check or changing the blockwise dropout schedules.


# CIFAR-100 ViT dropout scheduling

This is the transformer-side check of the scheduling idea. I am not treating the MLP calculation as a full ViT theory here; the question is simpler: does the same early-dropout bias appear when dropout is assigned block by block?


In [ ]:
from dropout_mft.paths import project_root

REPO_ROOT = project_root()

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import autocast, GradScaler
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from tqdm.auto import tqdm
import json
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

if device == 'cuda':
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    print(f"GPU: {torch.cuda.get_device_name(0)}")

USE_AMP = device == 'cuda'
AMP_DTYPE = torch.bfloat16 if (device == 'cuda' and torch.cuda.is_bf16_supported()) else torch.float16

In [ ]:
# Experiment settings

N_SIMULATIONS = 1

# Dropout field settings
H_BAR = 0.1
H_MAX = 0.2
# Schedule choices for this run - these mirror the MLP scheduling experiment,
# except the dropout field is assigned block by block instead of layer by layer.
SCHEDULES = [ 'none' ,'constant','reverse_step','reverse_linear']

# ViT architecture
EMBED_DIM = 128
DEPTH = 10
NUM_HEADS = 16
MLP_RATIO = 4.0
PATCH_SIZE= 4.0

# Training
EPOCHS = 75
BATCH_SIZE = 250
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.00
LR_MIN = 5e-6


# Evaluation cadence
EVAL_EVERY = 1

# Dataset
USE_FULL_DATASET = False
TRAIN_SIZE = None if USE_FULL_DATASET else 2000
TEST_SIZE = None if USE_FULL_DATASET else 5000

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')

from dropout_mft.schedules import DISPLAY_NAMES
from dropout_mft.style import SCHEDULE_COLORS, schedule_color

COLORS = SCHEDULE_COLORS

def get_color(s):
    return schedule_color(s)

def get_display_name(s):
    return DISPLAY_NAMES.get(s, s)

In [ ]:

# defining the different dropout schedules, as you can see most of them could be written in one line,
# but I wrote them out for clarity. In the transformer this is the same idea as the MLP:
# each block gets a dropout field, and the schedule decides where the regularization lives.

def get_dropout_schedule(sched, depth, h_bar, h_max=None):
    if sched == 'none': return [0.0] * depth
    if sched == 'constant': return [h_bar] * depth
    if sched== 'double': return [2*h_bar] * depth
    if sched== 'triple': return [3*h_bar] * depth
    if sched == 'big_step':
        n = max(1, int(depth / 3))
        return [3*h_bar] * n + [0.0] * (depth - n)

    if sched == 'linear':
        return [2*h_bar * i / (depth - 1) for i in range(depth)] if depth > 1 else [h_bar]
    if sched == 'reverse_linear':
        return [2*h_bar * (depth-1-i) / (depth-1) for i in range(depth)] if depth > 1 else [h_bar]
    if sched == 'step':
        h_max = h_max or 2*h_bar
        n = max(1, int(np.ceil(h_bar/h_max * depth)))
        return [0.0]*(depth-n) + [h_bar*depth/n]*n
    if sched == 'reverse_step':
        h_max = h_max or 2*h_bar
        n = max(1, int(np.ceil(h_bar/h_max * depth)))
        return [h_bar*depth/n]*n + [0.0]*(depth-n)
    raise ValueError(sched)

theory = {s: {'h_layers': get_dropout_schedule(s, DEPTH, H_BAR, H_MAX)} for s in SCHEDULES}
for s in SCHEDULES:
    print(f"{get_display_name(s):22s}: mean={np.mean(theory[s]['h_layers']):.4f}")

In [ ]:
# Vision Transformer definition

class PatchEmbed(nn.Module):
    def __init__(self, img_size=32, patch_size=PATCH_SIZE, embed_dim=384):
        super().__init__()
        self.proj = nn.Conv2d(3, embed_dim, patch_size, patch_size)
        self.num_patches = (img_size // patch_size) ** 2
    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)

class Attention(nn.Module):
    def __init__(self, dim, heads=8):
        super().__init__()
        self.heads, self.scale = heads, (dim // heads) ** -0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=False)
        self.proj = nn.Linear(dim, dim)
    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.heads, C // self.heads).permute(2,0,3,1,4)
        q, k, v = qkv.unbind(0)
        # Use scaled_dot_product_attention for speed on PyTorch 2.0+.
        x = torch.nn.functional.scaled_dot_product_attention(q, k, v)
        return self.proj(x.transpose(1,2).reshape(B, N, C))

class MLP(nn.Module):
    def __init__(self, dim, ratio=4.):
        super().__init__()
        h = int(dim * ratio)
        self.fc1, self.fc2 = nn.Linear(dim, h), nn.Linear(h, dim)
        self.act = nn.ReLU()
    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

class Block(nn.Module):
    def __init__(self, dim, heads, ratio=4., drop=0.):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.attn, self.mlp = Attention(dim, heads), MLP(dim, ratio)
        self.drop = nn.Dropout(drop)
    def forward(self, x):
        x = x + self.drop(self.attn(self.norm1(x)))
        x = x + self.drop(self.mlp(self.norm2(x)))
        return x

class ViT(nn.Module):
    def __init__(self, depth=12, dim=384, heads=6, ratio=4., h_layers=None):
        super().__init__()
        self.patch = PatchEmbed(32, 4, dim)
        n = self.patch.num_patches
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos = nn.Parameter(torch.zeros(1, n + 1, dim))
        h_layers = h_layers or [0.]*depth
        self.blocks = nn.ModuleList([Block(dim, heads, ratio, h_layers[i]) for i in range(depth)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, 100)
        nn.init.trunc_normal_(self.pos, std=.02)
        nn.init.trunc_normal_(self.cls, std=.02)
        self.apply(self._init)
    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
    def forward(self, x):
        x = self.patch(x)
        x = torch.cat([self.cls.expand(x.size(0),-1,-1), x], 1) + self.pos
        for blk in self.blocks: x = blk(x)
        return self.head(self.norm(x)[:,0])

print(f"Params: {sum(p.numel() for p in ViT(DEPTH, EMBED_DIM, NUM_HEADS, MLP_RATIO).parameters()):,}")

In [ ]:
# Keep the data resident on the GPU

def iterate_batches(x, y, bs, shuffle=True):
    n = x.shape[0]
    idx = torch.randperm(n, device=x.device) if shuffle else torch.arange(n, device=x.device)
    for i in range(0, n, bs):
        yield x[idx[i:i+bs]], y[idx[i:i+bs]]

def get_cifar100(train_size=None, test_size=None):
    mean = torch.tensor([0.5071, 0.4867, 0.4408]).view(1,3,1,1)
    std = torch.tensor([0.2675, 0.2565, 0.2761]).view(1,3,1,1)

    train = datasets.CIFAR100('./data', train=True, download=True)
    test = datasets.CIFAR100('./data', train=False, download=True)

    rng = np.random.RandomState(0)
    tr_idx = rng.choice(len(train.data), train_size, replace=False) if train_size else np.arange(len(train.data))
    te_idx = rng.choice(len(test.data), test_size, replace=False) if test_size else np.arange(len(test.data))

    x_tr = torch.from_numpy(train.data[tr_idx]).permute(0,3,1,2).float().div_(255)
    y_tr = torch.tensor(np.array(train.targets)[tr_idx])
    x_te = torch.from_numpy(test.data[te_idx]).permute(0,3,1,2).float().div_(255)
    y_te = torch.tensor(np.array(test.targets)[te_idx])

    if device == 'cuda':
        x_tr, y_tr, x_te, y_te = x_tr.cuda(), y_tr.cuda(), x_te.cuda(), y_te.cuda()
        mean, std = mean.cuda(), std.cuda()

    x_tr, x_te = (x_tr - mean) / std, (x_te - mean) / std
    print(f"Train: {x_tr.shape[0]}, Test: {x_te.shape[0]}")
    return (x_tr, y_tr), (x_te, y_te)

train_data, test_data = get_cifar100(TRAIN_SIZE, TEST_SIZE)

In [ ]:
# Training loop - nothing new here, just standard PyTorch training.
# The only real change is that every transformer block sees its assigned dropout probability.

def train_epoch(model, data, opt, crit, bs):
    model.train()
    x, y = data
    total, correct, loss_sum = 0, 0, 0.
    for xb, yb in iterate_batches(x, y, bs):
        opt.zero_grad(set_to_none=True)
        with autocast(dtype=AMP_DTYPE, enabled=USE_AMP):
            out = model(xb)
            loss = crit(out, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total += xb.size(0)
        correct += (out.argmax(1) == yb).sum().item()
        loss_sum += loss.item() * xb.size(0)
    return loss_sum/total, 100*correct/total

@torch.no_grad()
def evaluate(model, data, crit, bs):
    model.eval()
    x, y = data
    total, correct, loss_sum = 0, 0, 0.
    for xb, yb in iterate_batches(x, y, bs, shuffle=False):
        with autocast(dtype=AMP_DTYPE, enabled=USE_AMP):
            out = model(xb)
            loss = crit(out, yb)
        total += xb.size(0)
        correct += (out.argmax(1) == yb).sum().item()
        loss_sum += loss.item() * xb.size(0)
    return loss_sum/total, 100*correct/total

In [ ]:
# Run the experiment

def run_experiments(schedules, n_sims, epochs):
    results = {}

    for sched in schedules:
        print(f"\n{'='*50}")
        print(f"Schedule: {sched}")
        print(f"{'='*50}")

        h_layers = theory[sched]['h_layers']
        histories = []

        for sim in range(n_sims):
            seed = 42 + sim
            torch.manual_seed(seed)
            np.random.seed(seed)

            model = ViT(DEPTH, EMBED_DIM, NUM_HEADS, MLP_RATIO, h_layers).to(device)
            opt = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
            scheduler = CosineAnnealingLR(opt, T_max=epochs, eta_min=LR_MIN)
            crit = nn.CrossEntropyLoss()

            hist = {'train_acc': [], 'test_acc': [], 'train_loss': [], 'test_loss': []}
            best_test_acc, best_test_loss = 0., float('inf')

            pbar = tqdm(range(epochs), desc=f'{sched} sim {sim+1}/{n_sims}', leave=True)

            for ep in pbar:
                tr_loss, tr_acc = train_epoch(model, train_data, opt, crit, BATCH_SIZE)
                scheduler.step()

                if ep % EVAL_EVERY == 0 or ep == epochs - 1:
                    te_loss, te_acc = evaluate(model, test_data, crit, BATCH_SIZE)
                else:
                    te_loss = hist['test_loss'][-1] if hist['test_loss'] else 0
                    te_acc = hist['test_acc'][-1] if hist['test_acc'] else 0

                if te_acc > best_test_acc: best_test_acc = te_acc
                if te_loss < best_test_loss: best_test_loss = te_loss

                hist['train_acc'].append(tr_acc)
                hist['test_acc'].append(te_acc)
                hist['train_loss'].append(tr_loss)
                hist['test_loss'].append(te_loss)

                pbar.set_postfix({
                    'tr_L': f'{tr_loss:.3f}',
                    'tr_A': f'{tr_acc:.1f}%',
                    'te_L': f'{te_loss:.3f}',
                    'te_A': f'{te_acc:.1f}%',
                    'best': f'{best_test_acc:.1f}%'
                })

            histories.append(hist)
            del model, opt, scheduler
            torch.cuda.empty_cache()

        results[sched] = {
            'train_acc': np.array([h['train_acc'] for h in histories]),
            'test_acc': np.array([h['test_acc'] for h in histories]),
            'train_loss': np.array([h['train_loss'] for h in histories]),
            'test_loss': np.array([h['test_loss'] for h in histories]),
        }
        best = results[sched]['test_acc'].max(axis=1)
        print(f"{sched}: Best = {best.mean():.2f}% ± {best.std():.2f}%")

    return results

results = run_experiments(SCHEDULES, N_SIMULATIONS, EPOCHS)

In [ ]:
# Save after the runs finish
import json
import numpy as np

out = {s: {k: v.tolist() for k, v in results[s].items()} for s in SCHEDULES}
out['_meta'] = {
    'schedules': SCHEDULES,
    'n_simulations': N_SIMULATIONS,
    'epochs': EPOCHS,
    'h_bar': H_BAR,
    'h_max': H_MAX,
    'depth': DEPTH,
}
with open('vit_dropout_results.json', 'w') as f:
    json.dump(out, f)
print('Saved to vit_dropout_results.json')

# Optional Colab download (no-op when not running on Colab)
try:
    from google.colab import files
    files.download('vit_dropout_results.json')
except ImportError:
    pass

In [ ]:
# Plot the saved ViT scheduling curves
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams

from dropout_mft.paths import project_root

# Plot settings
START_EPOCH = 10          # crop out to see results more clearly
# Read the saved curves that ship with the repo (rerun the cells above to overwrite).
DATA_PATH = project_root() / "results" / "transformer" / "vit_dropout_results.json"

# Schedule labels and colors
from dropout_mft.schedules import DISPLAY_NAMES as names
from dropout_mft.style import SCHEDULE_COLORS as colors

# Plot defaults - the colors I like, feel free to change
rcParams.update({
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.linewidth": 0.9,
    "grid.linewidth": 0.7,
    "grid.alpha": 0.18,
    "lines.linewidth": 2.0,
    "figure.dpi": 130,
})

# Load saved results
with open(DATA_PATH) as f:
    data = json.load(f)

meta = data.pop("_meta", {})
schedules = meta.get("schedules", list(data.keys()))
results = {s: {k: np.array(v) for k, v in data[s].items()} for s in schedules}

# Build the figure
fig, axes = plt.subplots(2, 2, figsize=(12.5, 8.2), sharex=True)

panels = [
    ("train_loss", "Training loss"),
    ("test_loss",  "Test loss"),
    ("train_acc",  "Training accuracy (%)"),
    ("test_acc",   "Test accuracy (%)"),
]

legend_handles = []

for i, (key, title) in enumerate(panels):
    ax = axes[i // 2, i % 2]
    ax.set_title(title, pad=6)

    # Soften the legend box
    for spine in ax.spines.values():
        spine.set_alpha(0.25)
    ax.tick_params(width=0.8, length=3)

    for sched in schedules:
        arr = results[sched][key]
        mean, std = arr.mean(0), arr.std(0)

        # Crop the early epochs for readability
        epochs = np.arange(1, len(mean) + 1)
        keep = epochs >= START_EPOCH
        x, mean, std = epochs[keep], mean[keep], std[keep]

        color = colors.get(sched, "#333")
        lw = 2.4 if color == "#000000" else 2.0

        line, = ax.plot(x, mean, color=color, lw=lw, label=names.get(sched, sched))
        ax.fill_between(x, mean - std, mean + std, color=color, alpha=0.14, lw=0)

        if i == 0:
            legend_handles.append(line)

    ax.grid(True)
    if i >= 2:
        ax.set_xlabel("Epoch")

axes[0, 0].set_yscale("log")

fig.legend(
    legend_handles, [names.get(s, s) for s in schedules],
    loc="lower center", ncol=len(schedules), frameon=False,
    bbox_to_anchor=(0.5, -0.02),
)

plt.tight_layout(rect=(0, 0.06, 1, 1))
plt.savefig("vit_dropout_curves.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np  # Needed if you aren't already importing it

print(f"\n{'='*70}")
print(f"{'Schedule':<22} {'Best Test Acc':>20} {'Final Test Acc':>20}")
print(f"{'-'*70}")

for s in SCHEDULES:
    best = results[s]['test_acc'].max(axis=1)
    final = results[s]['test_acc'][:,-1]

    # Number of seeds
    n = len(best)

    # Standard error across seeds
    # ddof=1 gives the sample standard deviation across seeds
    best_sem = best.std(ddof=1) / (n ** 0.5)
    final_sem = final.std(ddof=1) / (n ** 0.5)

    print(f"{get_display_name(s):<22} {best.mean():>7.2f} ± {best_sem:<7.2f} {final.mean():>7.2f} ± {final_sem:<7.2f}")

print(f"{'='*70}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(SCHEDULES))
means = [results[s]['test_acc'].max(1).mean() for s in SCHEDULES]
stds = [results[s]['test_acc'].max(1).std() for s in SCHEDULES]
bars = ax.bar(x, means, .6, yerr=stds, capsize=4, color=[get_color(s) for s in SCHEDULES], edgecolor='k')
ax.set_xticks(x); ax.set_xticklabels([get_display_name(s) for s in SCHEDULES], rotation=25, ha='right')
ax.set_ylabel('Best Test Acc (%)')
ax.set_title('ViT Dropout Scheduling')
for b, m in zip(bars, means): ax.text(b.get_x()+b.get_width()/2, b.get_height()+.2, f'{m:.1f}', ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('vit_dropout_bar.png', dpi=150)
plt.show()

In [ ]:

out = {s: {k: v.tolist() for k,v in results[s].items()} for s in SCHEDULES}
with open('vit_dropout_results.json', 'w') as f: json.dump(out, f)
print('Saved to vit_dropout_results.json')